# Inhalt:
Klasse zur Erstellung, Standardisieren und Verwalten von Datasets (Train, Val, Test). 
# Input: 
Splitart (Train, Val, Test)
ein Config
Standardiesieren = True/False
# Output: 
Laden von Split und Embedding-Datei
Standardiesieren wenn standardize = True
Dimensionsprüfung über assert cfg.embed_dim

In [6]:
# %% ==0 Importe==
# Extern:

import pandas as pd
import torch
from tqdm.auto import tqdm

# Aus Projekt:
from aguirre.config import DATA_DIR, PROJECT_ROOT, Config, CACHE_DIR
from aguirre.data import read_fasta
from aguirre.esm_wrapper import ESMWrapper

cfg_8m   = Config(esm_model_name="esm2_t6_8M_UR50D",    embed_dim=320)
cfg_650m = Config(esm_model_name="esm2_t33_650M_UR50D", embed_dim=1280)

CACHE_DIR.mkdir(parents=True, exist_ok=True) # Noch nicht angelegt

In [ ]:
# %% ==1 Pfade==
TRACKED_DIR = PROJECT_ROOT / "wrath_of_god" / "data_tracked"
SPLIT_PATH  = TRACKED_DIR / "split_t075.csv"
FASTA_PATH = DATA_DIR / "all19K_unclustered.faa"

In [8]:
# %%Embedding-Schleife 
cfg = cfg_650m
df_split = pd.read_csv(filepath_or_buffer= SPLIT_PATH, sep= ",", names= ["ID", "Cluster", "Split"], header = 0)
df_split = df_split[["ID", "Split"]].copy()
ids, seqs = read_fasta(FASTA_PATH)
df_AMPs = pd.DataFrame({"ID": ids, "Sequenz": seqs})

splits = ["train", "val", "test"]
wrapper = ESMWrapper(cfg= cfg)

for split in splits:
    # Pfad
    EMBEDDING_PATH = CACHE_DIR / f"{split}-Set_{cfg.esm_model_name}_L{cfg.pep_max_len}.pt"

    maske_0 = df_split["Split"] == f"{split}"
    df_splitting = df_split[maske_0].copy()
    # Maskierung mit AMP_IDs im Train-Set
    df_amp_s = df_AMPs[df_AMPs["ID"].isin(df_splitting["ID"])].copy()

    df_s = df_amp_s.assign(laenge = df_amp_s["Sequenz"].str.len())
    df_sorted = df_s.sort_values("laenge", kind= "stable").reset_index(drop= True)
    #Sortierte Größen:
    seqs_sorted = df_sorted["Sequenz"].to_list()
    ids_sorted = df_sorted["ID"].to_list()
    lengths_sorted = df_sorted["laenge"].to_list()
    B = 16 #Batchgröße
    start_positionen = list(range(0, len(seqs_sorted), B)) #Sortieren führt dazu, dass Sequenzen gleicher Länge zusammenkommen (encode dimension für alle Embeddings abhängig von längster Sequenz)
    batches = [seqs_sorted[i:i+B] for i in start_positionen]

    # Batchschleife:
    N = len(seqs_sorted)

    emb_lengths = []
    emb_all = torch.empty(N, cfg.pep_max_len, cfg.embed_dim, dtype = torch.float32)
    pos = 0
    for b in tqdm(batches, desc=split): 
        EMBs, len_s = wrapper.encode(b) # Shape (B,L_max, D)
        emb_all[pos : pos + len(b)] = EMBs
        pos += len(b)
        emb_lengths.extend(len_s)
    #Guardians
    print(f"Anzahl von {split}-Set ist {N}")
    assert pos == N, f"{pos} statt {N} Zeilen gefüllt"
    assert emb_lengths == lengths_sorted, "Reihenfolge verrutscht"

    #Speichern
    torch.save({
        "emb": emb_all,
        "lengths": torch.tensor(lengths_sorted),
        "ids": ids_sorted,
    }, EMBEDDING_PATH) 



train:   0%|          | 0/970 [00:00<?, ?it/s]

Anzahl von train-Set ist 15507


val:   0%|          | 0/122 [00:00<?, ?it/s]

Anzahl von val-Set ist 1939


test:   0%|          | 0/122 [00:00<?, ?it/s]

Anzahl von test-Set ist 1938


In [10]:
#Test, ob Standards zu Train-Daten passen:


d = torch.load(CACHE_DIR / "train-Set_esm2_t6_8M_UR50D_L50.pt")
st = pd.read_csv(TRACKED_DIR / "standards_8M.csv")

mu    = torch.tensor(st["mean"].values, dtype=torch.float64)
sigma = torch.tensor(st["std"].values,  dtype=torch.float64)

emb, lengths = d["emb"], d["lengths"]
maske = torch.arange(emb.shape[1])[None, :] < lengths[:, None]    # (N, L_max)
z = (emb[maske].double() - mu) / sigma
print(f"Mittel = {z.mean():+.6f}   Std = {z.std():.6f}")

Mittel = +0.000000   Std = 1.000000
